# IDD: preprocessing and data loading

Use the cleaned **80/10/10** split: **5,507 train / 687 validation / 689 test** pairs.
This notebook explains the input pipeline for the trained U-Net/MobileNetV2 model.
It stops before model construction and training. Default reruns inspect training and validation only.
Stored outputs below are preserved historical evidence, including the earlier three-split preview; they were not regenerated in the 13 October review.

- Images: RGB, 512 x 512, float32, scaled to **[-1, 1]**.
- Masks: 512 x 512 x 1, float32, labels stay **0/1**. Never divide masks by 255.
- Training: shuffle pairs and flip image + mask horizontally together with 50% probability.
- Validation/test: no augmentation and a fixed order.
- Batch size: 4; seed: 42. Files are read and processed in memory.

The 110 empty-mask pairs have already been excluded. Original driving-sequence IDs are unavailable, so sequence separation cannot be guaranteed.


## 1. Imports and settings
Run cells from top to bottom. On this PC use the project's `.venv` Python environment.
Dependencies are listed in `requirements-preprocessing.txt`.

For Colab, copy the complete `splits` folder, including `pair_splits.csv`, to an accessible location and change `SPLITS_DIR` below. The code ignores the old absolute paths in the CSV and uses the current folder with pair IDs.


In [ ]:
"""TensorFlow input pipeline for the cleaned binary IDD road dataset.

Images: float32 [512, 512, 3], MobileNetV2 range [-1, 1].
Masks:  float32 [512, 512, 1], labels stay exactly 0 or 1.
Preprocessing happens in memory; PNG files are never rewritten.
"""

import csv
from pathlib import Path

import tensorflow as tf

IMAGE_SIZE = (512, 512)
SEED = 42
BATCH_SIZE = 4
SPLITS = ("train", "val", "test")

SPLITS_DIR = Path.cwd() / "dataset" / "splits"
# Example after mounting Google Drive in Colab:
# SPLITS_DIR = Path("/content/drive/MyDrive/ML Project/dataset/splits")

print("TensorFlow:", tf.__version__)
print("Dataset folder:", SPLITS_DIR.resolve())
if not (SPLITS_DIR / "pair_splits.csv").is_file():
    raise FileNotFoundError("Set SPLITS_DIR to the cleaned splits folder containing pair_splits.csv")


## 2. Read matching pairs
The numeric ID links `Image_123.png` to `Mask_123.png`. This check catches duplicate IDs, missing files, or files that are absent from the manifest.


In [ ]:
def read_pairs(splits_dir):
    """Use CSV IDs to pair files; reconstruct paths so the dataset is portable."""
    root = Path(splits_dir).expanduser().resolve()
    pairs = {split: [] for split in SPLITS}
    seen = set()
    with (root / "pair_splits.csv").open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if not {"id", "split"}.issubset(reader.fieldnames or []):
            raise ValueError("pair_splits.csv must contain id and split columns")
        for row in reader:
            pair_id, split = int(row["id"]), row["split"]
            if pair_id < 0 or pair_id in seen or split not in pairs:
                raise ValueError(f"Invalid or repeated pair: {row['id']}, {split}")
            seen.add(pair_id)
            image_path = root / split / "images" / f"Image_{pair_id}.png"
            mask_path = root / split / "masks" / f"Mask_{pair_id}.png"
            if not image_path.is_file() or not mask_path.is_file():
                raise FileNotFoundError(f"Missing image or mask for pair {pair_id}")
            pairs[split].append({"id": pair_id, "image": str(image_path), "mask": str(mask_path)})

    for split, records in pairs.items():
        records.sort(key=lambda row: row["id"])
        if not records:
            raise ValueError(f"The {split} split is empty")
        for folder, prefix in (("images", "Image"), ("masks", "Mask")):
            actual = {p.name for p in (root / split / folder).iterdir()}
            expected = {f"{prefix}_{row['id']}.png" for row in records}
            if actual != expected:
                raise ValueError(f"{split}/{folder} does not match the CSV manifest")
    return pairs

## 3. Decode, augment, and normalize
The files are already 512 x 512, so resizing is unnecessary. A shared flip decision preserves alignment. MobileNetV2 preprocessing affects only the image; the mask remains 0/1.

The cleaned dataset should have no empty masks. The loader raises an error if an empty mask or a label above 1 appears.


In [ ]:
def decode_pair(image_path, mask_path):
    """Read an RGB image and the original single-channel 0/1 labels."""
    image = tf.io.decode_png(tf.io.read_file(image_path), channels=3)
    mask = tf.io.decode_png(tf.io.read_file(mask_path), channels=1)
    image = tf.ensure_shape(image, [*IMAGE_SIZE, 3])
    mask = tf.ensure_shape(mask, [*IMAGE_SIZE, 1])
    # Conversion to float preserves 1 as 1; convert_image_dtype would rescale it.
    image = tf.cast(image, tf.float32)
    mask = tf.cast(mask, tf.float32)
    checks = [
        tf.debugging.assert_less_equal(tf.reduce_max(mask), 1.0, message="Mask contains labels above 1"),
        tf.debugging.assert_equal(tf.reduce_max(mask), 1.0, message="An empty mask entered the cleaned dataset"),
    ]
    with tf.control_dependencies(checks):
        return tf.identity(image), tf.identity(mask)

def augment_pair(image, mask, seed):
    """One shared random decision flips both tensors horizontally together."""
    flip = tf.random.stateless_uniform([], seed=seed) < 0.5
    return tf.cond(
        flip,
        lambda: (tf.image.flip_left_right(image), tf.image.flip_left_right(mask)),
        lambda: (image, mask),
    )

def normalize_pair(image, mask):
    """MobileNetV2 preprocessing changes images only; labels remain 0/1."""
    return tf.keras.applications.mobilenet_v2.preprocess_input(image), mask

## 4. Build batches
Only training data is shuffled and augmented. Random seeds change between passes while remaining reproducible from seed 42. The final smaller batch is retained.

The loader reads only the batches needed, rather than holding every decoded image in memory.


In [ ]:
def make_dataset(records, *, training=False, batch_size=BATCH_SIZE, seed=SEED, augment=True):
    """Build finite batches; preserve the final partial batch.

    Training shuffles file paths and uses fresh, reproducible random seeds on
    each pass. Validation/test retain their numeric ID order without augmentation.
    There is no full-dataset image cache, keeping memory usage bounded.
    """
    if batch_size < 1 or not records:
        raise ValueError("batch_size must be positive and records must not be empty")
    paths = tf.data.Dataset.from_tensor_slices((
        [row["image"] for row in records], [row["mask"] for row in records]
    ))
    if training:
        paths = paths.shuffle(len(records), seed=seed, reshuffle_each_iteration=True)

    if training and augment:
        random_seeds = tf.data.Dataset.random(seed=seed, rerandomize_each_iteration=True)
        paths = tf.data.Dataset.zip((paths, random_seeds))

        def load_training(pair_paths, random_seed):
            image, mask = decode_pair(*pair_paths)
            flip_seed = tf.stack([tf.cast(seed, tf.int32), tf.cast(random_seed, tf.int32)])
            image, mask = augment_pair(image, mask, flip_seed)
            return normalize_pair(image, mask)

        dataset = paths.map(load_training, num_parallel_calls=4, deterministic=True)
    else:
        dataset = paths.map(
            lambda image_path, mask_path: normalize_pair(*decode_pair(image_path, mask_path)),
            num_parallel_calls=4,
            deterministic=True,
        )

    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 4
    return dataset.with_options(options).batch(batch_size, drop_remainder=False).prefetch(1)

def build_datasets(splits_dir, batch_size=BATCH_SIZE, seed=SEED):
    pairs = read_pairs(splits_dir)
    datasets = {
        split: make_dataset(records, training=split == "train", batch_size=batch_size, seed=seed)
        for split, records in pairs.items()
    }
    return datasets, {split: len(records) for split, records in pairs.items()}

In [ ]:
tf.keras.utils.set_random_seed(SEED)
datasets, counts = build_datasets(SPLITS_DIR, batch_size=BATCH_SIZE, seed=SEED)
train_ds = datasets["train"]
val_ds = datasets["val"]
test_ds = datasets["test"]

for split in SPLITS:
    print(f"{split}: {counts[split]:,} pairs, {int(datasets[split].cardinality().numpy()):,} batches")


## 5. Check training and validation batches
Expected full-batch shapes: images `(4, 512, 512, 3)` and masks `(4, 512, 512, 1)`. Masks stay 0/1 and image values remain in [-1,1].

Test counts are checked from filenames/manifest. This cell does not decode test images; the locked final test report is already complete.


In [ ]:
import numpy as np

PREVIEW_SPLITS = ("train", "val")
batch_examples = {}
for split in PREVIEW_SPLITS:
    images, masks = next(iter(datasets[split]))
    x, y = images.numpy(), masks.numpy()
    assert images.shape == (BATCH_SIZE, 512, 512, 3)
    assert masks.shape == (BATCH_SIZE, 512, 512, 1)
    assert images.dtype == masks.dtype == tf.float32
    assert np.isfinite(x).all() and np.isfinite(y).all()
    assert x.min() >= -1 and x.max() <= 1
    assert set(np.unique(y)) == {0.0, 1.0}
    batch_examples[split] = (x, y)
    print(f"{split}: images {x.shape}, range [{x.min():.3f}, {x.max():.3f}]; masks {y.shape}, values {np.unique(y)}")
print("Training and validation batches passed; test contents were not read.")


## 6. Visually check alignment
Each group shows the image, its binary mask, and a green road overlay. Image normalization is reversed only for display. White mask pixels mean class 1; the actual labels stay 0/1.


In [ ]:
from PIL import Image, ImageDraw, ImageFont
from datetime import datetime

def save_preview(batches, path):
    tile, header, footer = 256, 38, 24
    panel_w, panel_h = tile * 3, tile + header + footer
    canvas = Image.new("RGB", (panel_w * 2, panel_h * len(batches)), "#111827")
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype("C:/Windows/Fonts/arial.ttf", 16)
    except OSError:
        font = ImageFont.load_default()
    for row, split in enumerate(batches):
        images, masks = batches[split]
        for column in range(2):
            x, y = column * panel_w, row * panel_h
            rgb = np.rint((images[column] + 1) * 127.5).clip(0, 255).astype(np.uint8)
            road = masks[column, ..., 0] == 1
            overlay = rgb.copy()
            overlay[road] = (rgb[road] * 0.55 + np.array([40, 240, 110]) * 0.45).astype(np.uint8)
            title = f"{split.upper()} | batch item {column + 1} | " + ("train flips enabled" if split == "train" else "no augmentation")
            draw.text((x + 8, y + 10), title, fill="white", font=font)
            views = [Image.fromarray(rgb), Image.fromarray(road.astype(np.uint8) * 255).convert("RGB"), Image.fromarray(overlay)]
            for i, (view, label) in enumerate(zip(views, ["Image: normalization reversed for display", "Mask: white = 1", "Road overlay"])):
                resampling = Image.Resampling.NEAREST if i == 1 else Image.Resampling.BILINEAR
                canvas.paste(view.resize((tile, tile), resampling), (x + i * tile, y + header))
                # Short labels keep the preview readable at 256 pixels per panel.
                label = ["Image (display values)", "Mask: white = 1", "Road overlay"][i]
                draw.text((x + i * tile + 8, y + header + tile + 3), label, fill="#cbd5e1", font=font)
    if Path(path).exists():
        raise FileExistsError("Choose a fresh preview path; earlier evidence is preserved")
    canvas.save(path)

preview_path = Path.cwd() / "preprocessing" / f"notebook_batch_preview_train_val_{datetime.now():%Y%m%d_%H%M%S_%f}.png"
preview_path.parent.mkdir(exist_ok=True)
save_preview(batch_examples, preview_path)
print("Preview saved:", preview_path)
Image.open(preview_path)


## Ready for the next stage
`train_ds`, `val_ds`, and `test_ds` are ready to feed a segmentation model. This notebook does not build or train a model. The model was trained separately and its locked final test report is already complete. Test data stays out of default previews and future development decisions. Do not add a second image normalization step to the future model.

The separate `check_preprocessing.py` tests flip alignment, seed reproducibility, pixel preservation, and the final partial batch; consult its current default scope before running.

References: [MobileNetV2 preprocessing](https://www.tensorflow.org/api_docs/python/tf/keras/applications/mobilenet_v2/preprocess_input), [TensorFlow segmentation tutorial](https://www.tensorflow.org/tutorials/images/segmentation).
